10/25/2026 Gottlieb

Before we recreate Vásquez-Farinelli's 21 asset simulation, let's start by implementing their algorithm on the 3 asset example (see section 2.1) simplified with one volatility shock.  

The paper's price model (Eq. 2) writes the return of an asset $\mu$ as such:
$$\frac{dX_\mu}{X_\mu} = \alpha_\mu\,dt + \sum_a\,\sigma_\mu^a\,dW_a$$

Where the symbols mean:
- $X_\mu$ is the price of asset $\mu$
- $\alpha_\mu$ is the drift (average growth rate)
- $W_a$ is some random shock $a$ (represented by Brownian motion)
- $\sigma_\mu^a$ is the strength of the asset's response to the shock

For the sake of our example we will consider one shock with identical exposure, and therefore we can omit the superscript $a$.

Our 3 assets are as follows:
- Asset 0 (cash account) with exposure to the shared shock $\sigma_0 = 0$
- Asset 1 (risky asset) with exposure $\sigma_1 = 1$
- Asset 2 (risky asset) with expsoure $\sigma_2 = 1$

Thus $\sigma = (0,1,1)^T$

(Eq. 20) of the paper's 3 asset example finds:
$$J = \frac{1}{\sqrt{2}\sqrt{\sigma_1^2 + \sigma_2^2 - \sigma_1\,\sigma_2}} \begin{pmatrix} \sigma_1 - \sigma_2 \\ \sigma_2 \\ -\sigma_1 \end{pmatrix}$$

Where we can think of $J_\mu$ as the dollar amount held in a particular asset (later in this notebook will see how such a vector is constructed). Then, substituting in our values:
$$J = \frac{1}{\sqrt{2}} \begin{pmatrix} 0 \\ 1 \\ -1 \end{pmatrix}$$

Which means we long Asset 1 and short an equal normalized amount of Asset 2. The factor $1/\sqrt{2}$ conditions the vector of to be unit length, it does not change which trade the vector represents.

The conditions of the null vector $J$ are required to satisfy from (Eq. 5):
$$\sum_\mu J_\mu = 0, \qquad \sum_\mu\,J_\mu\,\sigma_\mu^a = 0$$

From linear algebra these equations can be refactored:
$$J^T1=0, \qquad J^T\sigma^a=0$$

The first condition says that the positive and negative portfolio weights must balance to a zero-cost. The second says that the portfolio has no exposure to Brownian shocks (zero-risk).

For our purposes these conditions are satisified:
$$J^T1 = 0 + \frac{1}{\sqrt{2}} - \frac{1}{\sqrt{2}}= 0,
\qquad J^T\sigma = 0(0) + \frac{1}{\sqrt{2}}(1) - \frac{1}{\sqrt{2}}(1)= 0$$

In [1]:
import numpy as np
import pandas as pd

In [ ]:
N = 3
sigma = np.array([0.0, 1.0, 1.0])
J = np.array([0.0, 1.0, -1.0]) / np.sqrt(2)
ones = np.ones(N)

sum_of_weights = J @ ones
shock_exposure = J @ sigma
vector_length = np.linalg.norm(J)

print("J = ", np.round(J, 3))
print("Sum of weights = ", round(sum_of_weights, 3))
print("Shock exposure = ", round(shock_exposure, 3))
print("Length of J = ", round(vector_length, 3))

J =  [ 0.     0.707 -0.707]
Sum of weights =  0.0
Shock exposure =  0.0
Length of J =  1.0


Proposition 2.2 defines a covariance matrix:
$$\Omega_{\mu\nu} = \sum_a\sigma_\mu^a\sigma_\nu^a$$

From linear algebra we can refactor into $\sigma^a (\sigma^a)^T$. However we chose the simplifed case where there is only one shock, therefore the sum disappears $\Omega_{\mu\nu} = \sigma_\mu\sigma_\nu$ (equivalently $\sigma\sigma^T$). With the 3 asset example this becomes:
$$\Omega = \begin{pmatrix} 0 & 0 & 0 \\ 0 & \sigma_1^2 & \sigma_1\sigma_2 \\ 0 & \sigma_1\sigma_2 & \sigma_2^2 \end{pmatrix} = \begin{pmatrix} 0 & 0 & 0 \\ 0 & 1 & 1 \\ 0 & 1 & 1 \end{pmatrix}$$

We can then find the variance of the portfolio in direction of the null space vector $J$ which is found by $J^T\Omega\,J$. Further, because $J^T\sigma = 0$, we can expand the previous equation:
$$J^T\Omega\,J = J^T\sigma\sigma^T\,J = (J^T\sigma)^2 = 0$$

This is the linear algebraic statement which shows that the shared volatility cancels 

In [ ]:
Omega = np.outer(sigma, sigma)

assets = ["Cash", "Asset 1", "Asset 2"]
Omega_table = pd.DataFrame(Omega, index=assets, columns=assets)

display(Omega_table)

variance = J @ Omega @ J
print("Variance in direction J:", round(variance, 3))


,Cash,Asset 1,Asset 2
Cash,0.0,0.0,0.0
Asset 1,0.0,1.0,1.0
Asset 2,0.0,1.0,1.0


Variance in direction J: 0.0


Finding the null space of $\Omega$ is not enough. Cash itself has zero volatility, therefore it is a zero mode of $\Omega$ but does not satisfy the zero-cost condition. The paper constructs (Eq. 13) the matrix $G$ which incorporates the requirement $J^T\mathbf{1} = 0$. Proposition 2.2 defines:
$$G = \Omega - \frac{1}{N}(U\Omega + \Omega U) + \frac{1}{N^2}\operatorname{Tr}(U\Omega)U$$
Where $U$ is a matrix of all ones.

Though not explicitly formulated in the paper, there is a simpler way to understand this formula where we consider:
$$P = I - \frac{1}{N}U$$

Where $P$ projects onto the space orthogonal to the all-ones vector (the zero mode which fails the zero-cost). Expanding $P\Omega P$ produces exactly the paper's formula:
$$G=P\Omega P$$

To summarize we have established two facts:
1. The all-ones vectors is trivial, it is always a zero mode of $G$ 
2. All other zero modes of $G$ that are orthogonal to the all-ones vector are precisely vectors which satisfy (Eq. 5)

In [ ]:
U = np.ones((N, N))

# Eq. 13
G = Omega - (U @ Omega + Omega @ U) / N + np.trace(U @ Omega) * U / N**2

# Equivalent expression
identity = np.eye(N)
P = identity - U / N
G_eq = P @ Omega @ P

G_table = pd.DataFrame(G, index=assets, columns=assets)
display(G_table.round(3))

print("Eq. 13 equals the equivalent expression:", np.allclose(G, G_eq))
print("The all-ones vector is a trivial zero mode of G:", np.round(G @ ones, 3))
print("The J vector is a non-trival zero mode of G:", np.round(G @ J, 10))
print("The J vector is zero-risk:", round(J @ G @ J, 10))

,Cash,Asset 1,Asset 2
Cash,0.444,-0.222,-0.222
Asset 1,-0.222,0.111,0.111
Asset 2,-0.222,0.111,0.111


Eq. 13 equals the equivalent expression: True
The all-ones vector is a trivial zero mode of G: [0. 0. 0.]
J is a non-trival zero mode of G: [0. 0. 0.]
J is zero-risk: 0.0


Section 4.1 defines algorithm to estimate the null direction vectors. This is found by the eigenvectors of $\hat G$ (Eq. 86) where the hat signifies it is an estimate:
$$\hat G \hat J = \lambda \hat J$$

The corresponding eigenvalue tells us the amount of variation associated with that particular direciton. A eigenvalue of zero is thus an exact null direction in the theoretical model. Our exact $G$ from the example has two eigenvalues of zero. One for the trivial all-ones direction, and one for the direction $J$.

When multiple eigenvectors share the same eigenvalue, the individual eigenvectors are not unique. The function `np.linalg.eigh` can return any orthonormal basis spanning the shared eigenvalue space. For our example, this may be the two directions we expect to find, or some combination. This is valid because only the combined eigenspace is unique. The paper addresses this by labeling the all-ones direction $J^0$ and choosing the remaining zero modes orthogonal.

In [ ]:
eigenvalues, eigenvectors = np.linalg.eigh(G)

print("Eigenvalues:", np.round(eigenvalues, 3))
display(pd.DataFrame(eigenvectors, index=assets, columns=["Direction 0", "Direction 1", "Direction 2"],).round(3))

Exact eigenvalues: [0.    0.    0.667]


,Direction 0,Direction 1,Direction 2
Cash,-0.072,0.573,0.816
Asset 1,0.630,0.661,-0.408
Asset 2,-0.773,0.485,-0.408


The algorithm estimates $\hat \Omega$ from a finite window of log returns (Eq. 93):
$$\hat{\Omega}_{\mu\nu}(t) = \frac{1}{L}\sum_{i=0}^{L-1} \log\!\left(\frac{X_\mu(t-i)}{X_\mu(t-i-1)}\right)\log\!\left(\frac{X_\nu(t-i)}{X_\nu(t-i-1)}\right) - \frac{1}{L^2}\sum_{i,j=0}^{L-1} \log\!\left(\frac{X_\mu(t-i)}{X_\mu(t-i-1)}\right)\log\!\left(\frac{X_\nu(t-j)}{X_\nu(t-j-1)}\right)$$

With this we have now found a portfolio direction which minimizes volatility (an estimate for the null direction). We have not yet found an arbitrage profit. The implementation of the arbitrage detection algorithm outlined by Vásquez and Farinelli is the subject of the next notebook.